<a href="https://colab.research.google.com/github/codewithsraj/sonar-ai-project/blob/main/SONAR_Detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install gradio


In [17]:
# ============================================
# 🚢 SONAR AI PLATFORM
# LOGIN + SIGNUP + ADVANCED ADMIN PANEL
# Google Colab Ready
# ============================================

!pip install gradio pandas scikit-learn matplotlib -q

import gradio as gr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# ============================================
# LOAD DATASET
# ============================================

df = pd.read_csv("Copy of sonar data.csv", header=None)

columns = [f"Echo_Signal_{i}" for i in range(1,61)] + ["Detected_Object"]
df.columns = columns

X = df.iloc[:, :-1]
y = df.iloc[:, -1]

# ============================================
# TRAIN MODEL
# ============================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

model = RandomForestClassifier(n_estimators=300, random_state=42)
model.fit(X_train, y_train)

pred = model.predict(X_test)
acc = accuracy_score(y_test, pred)

# ============================================
# USER DATABASE
# ============================================

users = {
    "admin": {"password": "1234", "role": "Admin"}
}

# ============================================
# SIGNUP
# ============================================

def signup(new_user, new_pass):
    if new_user in users:
        return "❌ Username already exists"

    users[new_user] = {
        "password": new_pass,
        "role": "User"
    }

    return "✅ Signup Successful! Now Login."

# ============================================
# LOGIN
# ============================================

def login(username, password):
    if username in users and users[username]["password"] == password:

        role = users[username]["role"]

        return (
            f"✅ Login Successful | Role: {role}",
            gr.update(visible=True),
            gr.update(visible=True),
            gr.update(visible=(role=="Admin")),
            gr.update(visible=(role=="Admin"))
        )

    return (
        "❌ Invalid Login",
        gr.update(visible=False),
        gr.update(visible=False),
        gr.update(visible=False),
        gr.update(visible=False)
    )

# ============================================
# PREDICT
# ============================================

def predict_sonar(*inputs):
    data = np.array(inputs).reshape(1,-1)
    result = model.predict(data)[0]

    if result == 'R':
        return "🪨 ROCK DETECTED", "Safe Object"
    return "💣 MINE DETECTED", "Dangerous Object"

# ============================================
# RANDOM FILL
# ============================================

def random_fill():
    return [round(random.uniform(0,1),2) for _ in range(60)]

# ============================================
# GRAPH
# ============================================

def graph():
    models = ["Logistic","KNN","SVM","RF"]
    scores = [78,86,91,round(acc*100,2)]

    fig, ax = plt.subplots(figsize=(8,4))
    ax.bar(models,scores)
    ax.set_title("Accuracy Comparison")
    ax.set_ylabel("Accuracy %")
    return fig

# ============================================
# ADMIN FUNCTIONS
# ============================================

def total_users():
    return f"👥 Total Users: {len(users)}"

def show_users():
    data = []
    for u in users:
        data.append([u, users[u]["role"]])
    return pd.DataFrame(data, columns=["Username","Role"])

def delete_user(username):
    if username == "admin":
        return "❌ Admin cannot be deleted"

    if username in users:
        del users[username]
        return f"✅ {username} deleted"

    return "❌ User not found"

def promote_user(username):
    if username in users:
        users[username]["role"] = "Admin"
        return f"🚀 {username} promoted to Admin"

    return "❌ User not found"

# ============================================
# INPUTS
# ============================================

inputs = [gr.Number(label=f"Echo_Signal_{i+1}", value=0) for i in range(60)]

# ============================================
# UI
# ============================================

with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
    # 🚢 SONAR AI SECURE PLATFORM
    ### Login + Signup + Admin Control Center
    """)

    # SIGNUP TAB
    with gr.Tab("📝 Signup"):
        su_user = gr.Textbox(label="Create Username")
        su_pass = gr.Textbox(label="Create Password", type="password")
        su_status = gr.Textbox(label="Signup Status")

        gr.Button("Signup").click(
            signup,
            [su_user, su_pass],
            su_status
        )

    # LOGIN TAB UPDATED
    with gr.Tab("🔐 Login"):
        username = gr.Textbox(label="Username", placeholder="Enter username")
        password = gr.Textbox(label="Password", type="password", placeholder="Enter password")
        status = gr.Textbox(label="Login Status", interactive=False)

        login_btn = gr.Button("Login", variant="primary")

    # PREDICT TAB
    with gr.Tab("🎯 Predict", visible=False) as predict_tab:

        for i in range(0,60,10):
            with gr.Row():
                for j in range(10):
                    inputs[i+j].render()

        p1 = gr.Textbox(label="Prediction")
        p2 = gr.Textbox(label="Status")

        gr.Button("🚀 Predict").click(
            predict_sonar,
            inputs=inputs,
            outputs=[p1,p2]
        )

        gr.Button("🎲 Random Fill").click(
            random_fill,
            outputs=inputs
        )

    # ANALYTICS TAB
    with gr.Tab("📊 Analytics", visible=False) as graph_tab:
        plot = gr.Plot()

        gr.Button("Show Graph").click(
            graph,
            outputs=plot
        )

    # ADMIN TAB
    with gr.Tab("⚙ Admin Panel", visible=False) as admin_tab:

        gr.Markdown("## ⚙ Admin Control Center")

        total_box = gr.Textbox(label="Platform Stats")
        gr.Button("📊 Total Users").click(
            total_users,
            outputs=total_box
        )

        user_table = gr.Dataframe(label="Registered Users")
        gr.Button("👥 Show Users").click(
            show_users,
            outputs=user_table
        )

        del_user = gr.Textbox(label="Delete Username")
        del_status = gr.Textbox(label="Delete Status")

        gr.Button("🗑 Delete User").click(
            delete_user,
            del_user,
            del_status
        )

        pro_user = gr.Textbox(label="Promote Username")
        pro_status = gr.Textbox(label="Promotion Status")

        gr.Button("🚀 Promote to Admin").click(
            promote_user,
            pro_user,
            pro_status
        )

    # ABOUT TAB
    with gr.Tab("ℹ About", visible=False) as about_tab:
        gr.Markdown(f"""
        Accuracy: {round(acc*100,2)}%

        Roles:
        - Admin = Full Access
        - User = Prediction Only
        - User can signup manually
        """)

    # LOGIN BUTTON CONNECT
    login_btn.click(
        login,
        [username,password],
        [status,predict_tab,about_tab,admin_tab,graph_tab]
    )

demo.launch(share=True)

/tmp/ipykernel_31787/921994594.py:166: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://42ebeb441c43df2706.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
